# 🧠 NEURO-CUT // Phase 3: PPO Reinforcement Learning Policy Training (Google Colab Worker)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmanM006/neurocut/blob/main/notebooks/train_ppo_colab.ipynb)

This notebook runs the **Cloud PPO RL Training Worker** for **Neuro-Cut** (Google Cloud Agentic Cinema Hackathon).

### Key Highlights:
1. **Zero Load on Local Machine**: All policy gradient updates, state transitions, and rollouts run in Google Colab's cloud environment.
2. **Real ClickHouse Cloud Integration**: Streams episode attempts and audience retention curves straight into ClickHouse Cloud (`fwybcmwtlx.asia-southeast1.gcp.clickhouse.cloud`).
3. **Live Dashboard Sync**: As this notebook runs, your local Next.js frontend (**Panel D: PPO RL Policy Training**) dynamically renders the live 20-episode rolling average curve via ClickHouse SQL window functions!
4. **Action Masking & Deterministic Evaluation**: Constrains policy to valid cinematic edits and tests against the **Beam Search Baseline (`0.6730`)**.

### Step 1: Install System Dependencies & Clone Repository

In [ ]:
!apt-get update -qq && apt-get install -y -qq ffmpeg
!pip install -q clickhouse-connect pydantic pydantic-settings opencv-python pillow torch
!rm -rf /content/neurocut && git clone https://github.com/AmanM006/neurocut.git /content/neurocut
%cd /content/neurocut

import sys
if '/content/neurocut' not in sys.path:
    sys.path.insert(0, '/content/neurocut')
print('>>> Environment ready! Repository loaded at /content/neurocut')

### Step 2: Connect to Live ClickHouse Cloud & Verify Baseline Target

In [ ]:
import clickhouse_connect

print('>>> Connecting to ClickHouse Cloud...')
ch_client = clickhouse_connect.get_client(
    host='fwybcmwtlx.asia-southeast1.gcp.clickhouse.cloud',
    port=8443,
    user='default',
    password='2~gQ9oPIQ3CEU',
    secure=True
)
rows = ch_client.query('SELECT count() FROM default.edit_attempts').result_set[0][0]
print(f'Connected! Existing edit attempts in ClickHouse: {rows}')

baseline_row = ch_client.query("""
    SELECT episode_id, max(reward) as r 
    FROM default.edit_attempts 
    WHERE episode_id = 'beam_search_baseline' 
    GROUP BY episode_id
""").result_set

if baseline_row:
    print(f'🎯 Target Baseline to Beat: {baseline_row[0][0]} -> Reward: {baseline_row[0][1]:.4f}')
else:
    print('🎯 Target Baseline to Beat: 0.6730 (Beam Search Baseline)')

### Step 3: Run PPO Curriculum Training (Configurable Episodes)
Set `TOTAL_EPISODES` below (e.g. 100, 200, 300). Each episode tests action-masked editing decisions and logs directly into ClickHouse Cloud.

In [ ]:
import os
import time
import numpy as np
from pathlib import Path

from backend.optimizer.ppo_agent import PPOAgent
from backend.clickhouse.client import clickhouse_client
from backend.clickhouse.reward_queries import compute_clickhouse_reward

# Configurable Training Budget
TOTAL_EPISODES = 200        # Recommended: 200 (~15 min). Fast test: 100 (~7 min). Deep: 300-500 (~25-45 min)
STEPS_PER_EPISODE = 4
CHECKPOINT_INTERVAL = 25

MODELS_DIR = Path('/content/neurocut/backend/models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print('=' * 70)
print('      NEURO-CUT // PPO REINFORCEMENT LEARNING COLAB WORKER')
print(f'      Episodes: {TOTAL_EPISODES} | Steps/Ep: {STEPS_PER_EPISODE} | Checkpoint: Every {CHECKPOINT_INTERVAL} eps')
print('=' * 70)

agent = PPOAgent(episode_id='ppo_colab_train_ep_1')
agent.scorer.gemini_client = None  # Fast deterministic scoring for RL rollouts

best_eval_reward = -999.0
best_checkpoint_path = str(MODELS_DIR / 'ppo_best.npz')
history = []
start_time = time.time()

for ep in range(1, TOTAL_EPISODES + 1):
    ep_id = f'ppo_colab_train_ep_{ep}'
    initial_reward = agent.reset_episode(ep_id)
    ep_rewards = [initial_reward]

    for step in range(STEPS_PER_EPISODE):
        step_res = agent.optimize_step(compile_video=False, deterministic=False)
        ep_rewards.append(step_res['reward'])

    train_metrics = agent.train_step()
    final_ep_reward = max(ep_rewards)
    history.append(final_ep_reward)

    rolling_avg = np.mean(history[-20:])

    if ep % 5 == 0 or ep == 1 or ep == TOTAL_EPISODES:
        elapsed = time.time() - start_time
        print(f'Ep {ep:>3}/{TOTAL_EPISODES} | Final Reward: {final_ep_reward:.4f} | '
              f'Rolling Avg (20): {rolling_avg:.4f} | Loss: {train_metrics.get("loss", 0):.4f} | '
              f'Time: {elapsed:.1f}s')

    if ep % CHECKPOINT_INTERVAL == 0 or ep == TOTAL_EPISODES:
        ckpt_path = str(MODELS_DIR / f'ppo_checkpoint_ep{ep}.npz')
        agent.save_checkpoint(ckpt_path)

        # Deterministic evaluation on fresh episode
        eval_ep_id = f'ppo_colab_eval_ep{ep}'
        agent.reset_episode(eval_ep_id)
        eval_rewards = []
        for _ in range(STEPS_PER_EPISODE):
            ev = agent.optimize_step(compile_video=False, deterministic=True)
            eval_rewards.append(ev['reward'])
        eval_final = max(eval_rewards) if eval_rewards else 0.0
        print(f'  >>> Checkpoint Ep {ep} Deterministic Eval: {eval_final:.4f} (Previous Best: {best_eval_reward:.4f})')

        if eval_final > best_eval_reward:
            best_eval_reward = eval_final
            agent.save_checkpoint(best_checkpoint_path)
            print(f'  *** NEW BEST MODEL SAVED! Eval: {best_eval_reward:.4f} ***')

        agent.buffer.clear()

total_time = time.time() - start_time
print('\n' + '=' * 70)
print(f'      PPO TRAINING COMPLETED IN {total_time:.1f}s ({total_time/60:.1f} min)')
print(f'      Best Deterministic Eval Reward: {best_eval_reward:.4f}')
print('=' * 70)

### Step 4: Run Frozen-Policy Benchmark on `ppo_final_eval`
Evaluates the final checkpoint deterministically and logs to `ppo_final_eval` in ClickHouse Cloud.

In [ ]:
print('\n[Step 4] Executing Final Frozen Policy Evaluation on \'ppo_final_eval\'...')
eval_agent = PPOAgent(episode_id='ppo_final_eval')
eval_agent.scorer.gemini_client = None
loaded = eval_agent.load_checkpoint(best_checkpoint_path)
print(f'  * Best checkpoint loaded ({best_checkpoint_path}): {loaded}')

eval_init = eval_agent.reset_episode('ppo_final_eval')
final_rewards = [eval_init]

for s in range(STEPS_PER_EPISODE):
    ev_step = eval_agent.optimize_step(compile_video=(s == STEPS_PER_EPISODE - 1), deterministic=True)
    final_rewards.append(ev_step['reward'])
    print(f'  Eval Step #{s+1} | Action: {ev_step["action"]:^14} on {str(ev_step["target_clip_id"]):^24} | '
          f'Reward: {ev_step["reward"]:.4f} | Verdict: {ev_step["verdict"]}')

ppo_final_reward = max(final_rewards)
print(f'\n>>> PPO Frozen Policy Final Evaluated Reward: {ppo_final_reward:.4f}')

print('\n' + '=' * 70)
print('      CLICKHOUSE CLOUD HEAD-TO-HEAD BENCHMARK')
print('=' * 70)
query = '''
SELECT episode_id, max(reward) as final_reward 
FROM default.edit_attempts 
WHERE episode_id IN ('beam_search_baseline', 'ppo_final_eval')
GROUP BY episode_id
ORDER BY final_reward DESC
'''
rows = clickhouse_client.query(query)
for r in rows:
    tag = '  <-- CURRENT CHAMPION' if r['final_reward'] == max(x['final_reward'] for x in rows) else ''
    print(f'  * Episode: {r["episode_id"]:^24} | Final Reward: {r["final_reward"]:.4f}{tag}')

### Step 5: Visualize the 20-Episode Rolling Learning Curve

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(11, 5), dpi=120)
episodes = np.arange(1, len(history) + 1)
rolling = [np.mean(history[max(0, i-20):i+1]) for i in range(len(history))]

plt.plot(episodes, history, alpha=0.25, color='cyan', label='Episode Raw Reward')
plt.plot(episodes, rolling, color='#00d4ff', linewidth=2.5, label='20-Episode Rolling Average')
plt.axhline(0.6730, color='#ffaa00', linestyle='--', linewidth=2, label='Beam Search Baseline (0.6730)')
plt.axhline(0.5000, color='#888888', linestyle=':', linewidth=1.5, label='Rough Cut Initial (0.5000)')

plt.title('NEURO-CUT: PPO Policy Learning Progression (ClickHouse Retention Reward)', fontsize=13, fontweight='bold')
plt.xlabel('Episode', fontsize=11)
plt.ylabel('Scalar Retention Reward', fontsize=11)
plt.legend(loc='lower right')
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

### Step 6: Download Best Trained Weights (`ppo_best.npz`)
Run this cell to download the trained checkpoint directly to your local computer.

In [ ]:
from google.colab import files
if os.path.exists(best_checkpoint_path):
    print(f'>>> Downloading {best_checkpoint_path}...')
    files.download(best_checkpoint_path)
else:
    print('Checkpoint not found!')